# ComSeg on the Xenium breast cancer dataset (xenium_2)

This notebook runs **ComSeg** on the Xenium human breast cancer dataset by calling the shared driver script `comseg_xenium_4.py` with unified parameters.

### Why this dataset was re-run

In the previous run, the cached input `tx.csv` for the breast dataset was byte-for-byte identical to that of the lymph node dataset (8,447,078 bytes, md5 `ca1e2d346f6f4385`). The earlier breast run had therefore used the lymph node input, and **all** metrics of ComSeg on xenium_2 in the summary table were invalid, not only the detection metrics.

### Why this notebook does not preprocess the data itself

The `DATASETS[*]["root"]` entries of `comseg_xenium_4.py` point to the raw Xenium directories. The driver script reads `transcripts.parquet` itself, crops the ROI according to `--roi-size`, splits it into tiles, and applies quality-value filtering. Preparing a separate CSV here would duplicate that work and reintroduce the risk of using the wrong input file. This notebook therefore does only two things:

1. Make sure that the `DATASETS` dictionary in `comseg_xenium_4.py` contains an entry for the breast dataset, and add it if it is missing.
2. Call the driver script with unified parameters.

### ROI and tiling

The ROI is 5,000 pixels wide (0.2125 µm per pixel, i.e. 1,062.5 µm square). An independent check showed that this ROI contains about 5,557 reference cells with 331.9 transcripts per cell, which is large enough and matches the region used for the other methods on xenium_2. With 2,000-pixel tiles, the ROI is split into 9 tiles of about 200,000 transcripts each, eight times fewer than the 1.57 million transcripts per tile of the mouse brain dataset (about 11 hours).

## 1. Configuration

Edit the paths and parameters below. `STAGE` selects which part of the pipeline is run:

- `"prep"`: crop the ROI and split it into tiles
- `"run"`: run ComSeg on each tile
- `"merge"`: merge the tile results
- `"all"`: run all of the above in order
- `"report"`: summarize an existing result

Set `CHECK_ONLY = True` to verify the inputs and print the command without running anything.

In [ ]:
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

BASE        = Path("yourpath/comseg/comseg_xenium_4.py")   # shared ComSeg driver script
BREAST_ROOT = Path("yourpath/dataset/xenium_breast")
DATASET_KEY = "breast"

STAGE      = "all"     # one of "prep", "run", "merge", "all", "report"
CHECK_ONLY = True      # True: check the configuration only; False: actually run

ROI_SIZE              = 5000    # pixels
ROI_ORIGIN            = None    # (y0, x0) in pixels, or None to centre the ROI on the image
TILE_PX               = 2000
TILE_OVERLAP_PX       = 150
MEAN_CELL_DIAMETER_UM = 12.0
MAX_CELL_RADIUS_UM    = 20.0
QV_MIN                = 20.0
EXTRA_ARGS            = []      # additional arguments passed through to comseg_xenium_4.py

PIXEL_SIZE_UM = 0.2125          # Xenium morphology image pixel size

ENTRY = f'    "{DATASET_KEY}": {{"root": "{BREAST_ROOT}", "tag": "{DATASET_KEY}"}},\n'


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)

## 2. Check the inputs

The candidate file locations below are kept consistent with the `*_CANDIDATES` lists in `comseg_xenium_4.py`.

In [ ]:
def check_inputs():
    ok = True
    log(f"Data directory: {BREAST_ROOT}")
    if not BREAST_ROOT.is_dir():
        log("  [MISSING] directory does not exist")
        return False
    required = {
        "transcripts": ["outs/transcripts.parquet", "transcripts.parquet"],
        "experiment metadata": ["outs/experiment.xenium", "experiment.xenium"],
        "nucleus boundaries": ["outs/nucleus_boundaries.parquet",
                               "nucleus_boundaries.parquet"],
        "morphology image": ["outs/morphology_focus/morphology_focus_0000.ome.tif",
                             "outs/morphology_focus.ome.tif",
                             "outs/morphology_mip.ome.tif",
                             "outs/morphology.ome.tif",
                             "morphology_focus/morphology_focus_0000.ome.tif",
                             "morphology.ome.tif"],
    }
    for name, candidates in required.items():
        hit = next((c for c in candidates if (BREAST_ROOT / c).exists()), None)
        if hit:
            size_gb = (BREAST_ROOT / hit).stat().st_size / 1e9
            log(f"  [OK]      {name:20s} {hit}  ({size_gb:.2f} GB)")
        else:
            log(f"  [MISSING] {name:20s} not found, tried: {candidates}")
            ok = False
    gt = BREAST_ROOT / "cells.parquet"
    if gt.exists():
        try:
            import pandas as pd
            n = len(pd.read_parquet(gt, columns=["cell_id"]))
            log(f"  [OK]      reference cells      cells.parquet  {n:,} cells (whole section)")
        except Exception as e:
            log(f"  [WARN]    cells.parquet exists but could not be read: {e}")
    return ok


log("=" * 70)
log("ComSeg on xenium_2 (breast)")
log("=" * 70)
if not check_inputs():
    raise RuntimeError("The inputs are incomplete. Provide the missing files before running.")

## 3. Register the breast dataset in the driver script

This step inserts the breast entry into the `DATASETS` dictionary of `comseg_xenium_4.py` if it is not already there. It is idempotent, and a timestamped backup of the driver script is written before any change. With `CHECK_ONLY = True`, the entry is only printed and not written.

In [ ]:
def ensure_dataset_entry(dry=False):
    if not BASE.exists():
        raise FileNotFoundError(f"Driver script not found: {BASE}")
    src = BASE.read_text()
    if re.search(rf'^\s*"{DATASET_KEY}"\s*:', src, re.M):
        log(f'DATASETS already contains an entry for "{DATASET_KEY}"')
        return True
    m = re.search(r"^DATASETS\s*=\s*\{\s*$", src, re.M)
    if not m:
        log("[ERROR] Could not find the line 'DATASETS = {'. Add this entry to DATASETS manually:")
        log("  " + ENTRY.strip())
        return False
    log(f'DATASETS has no entry for "{DATASET_KEY}". The following entry will be inserted:')
    log("  " + ENTRY.strip())
    if dry:
        log("  (check only, nothing written)")
        return False
    backup = BASE.with_suffix(f".py.bak_{time.strftime('%m%d_%H%M')}")
    shutil.copy2(BASE, backup)
    insert_at = m.end() + 1
    BASE.write_text(src[:insert_at] + ENTRY + src[insert_at:])
    log(f"  [OK] Entry inserted. Backup written to {backup.name}")
    return True


entry_ok = ensure_dataset_entry(dry=CHECK_ONLY)
if not entry_ok and not CHECK_ONLY:
    raise RuntimeError("The DATASETS entry could not be inserted automatically. Add it manually as shown above.")

## 4. Build the command

In [ ]:
n_tiles = ((ROI_SIZE + TILE_PX - 1) // TILE_PX) ** 2
out_dir = f"yourpath/comseg_xenium_{DATASET_KEY}_roi{ROI_SIZE}"

log(f"ROI      {ROI_SIZE} px x {PIXEL_SIZE_UM} um/px = {ROI_SIZE * PIXEL_SIZE_UM:.1f} um square")
log(f"Tiles    {TILE_PX} px, overlap {TILE_OVERLAP_PX} px -> about {n_tiles} tiles")
log(f"Cells    mean diameter {MEAN_CELL_DIAMETER_UM} um, maximum radius {MAX_CELL_RADIUS_UM} um")
log(f"QV min   {QV_MIN}")
log(f"Output   {out_dir}")
log("Reference: an independent check found about 5,557 reference cells and 331.9 transcripts per cell in this ROI")

cmd = [sys.executable, str(BASE),
       "--dataset", DATASET_KEY,
       "--stage", STAGE,
       "--roi-size", str(ROI_SIZE),
       "--tile-px", str(TILE_PX),
       "--tile-overlap-px", str(TILE_OVERLAP_PX),
       "--mean-cell-diameter-um", str(MEAN_CELL_DIAMETER_UM),
       "--max-cell-radius-um", str(MAX_CELL_RADIUS_UM),
       "--qv-min", str(QV_MIN)]
if ROI_ORIGIN:
    cmd += ["--roi-origin", str(ROI_ORIGIN[0]), str(ROI_ORIGIN[1])]
cmd += [x for x in EXTRA_ARGS if x != "--"]

log("$ " + " ".join(cmd))

## 5. Run ComSeg

The driver script is run as a subprocess, and its output is streamed into this notebook line by line.

**Note on resources.** ComSeg can use a large amount of memory on big tiles. If other large jobs are running on the same machine, wait until they finish before starting this one.

In [ ]:
if CHECK_ONLY:
    log("CHECK_ONLY is True, so nothing was run.")
    log('To run, set CHECK_ONLY = False and STAGE = "prep" (or "all"), then re-run the notebook.')
else:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"comseg_xenium_4.py exited with code {returncode}")
    log("Finished successfully.")